<a href="https://colab.research.google.com/github/ilfpns/PhoSem/blob/main/Phosem_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 폰트 설정 및 라이브러리 추가
import os
import re
import glob
import shutil
import random

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import torchvision.transforms.functional as TF

from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

!apt-get -qq install fonts-nanum > /dev/null

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

print("폰트 설정 완료:", plt.rcParams['font.family'])

In [ ]:
# @title 파일 불러오기
if not any('Nanum' in f.name for f in fm.fontManager.ttflist):
    os.system("apt-get -qq install fonts-nanum > /dev/null 2>&1")
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

from google.colab import drive
drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/원본2000"
DATA_DIR = "/content/dataset"

marker_path = os.path.join(DATA_DIR, ".src_marker")
need_copy = True
if os.path.exists(DATA_DIR) and os.path.exists(marker_path):
    with open(marker_path) as f:
        prev_src = f.read().strip()
    if prev_src == SRC and len(os.listdir(DATA_DIR)) > 1:
        need_copy = False

if need_copy:
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    os.makedirs(DATA_DIR, exist_ok=True)
    IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp")
    copied, seen_names = 0, set()
    for root, dirs, files in os.walk(SRC, followlinks=False):
        dirs[:] = [d for d in dirs if not os.path.islink(os.path.join(root, d))]
        for f in files:
            if f.lower().endswith(IMG_EXTS) and f not in seen_names:
                shutil.copy2(os.path.join(root, f), os.path.join(DATA_DIR, f))
                seen_names.add(f)
                copied += 1
    with open(marker_path, "w") as f:
        f.write(SRC)
    print(f"복사 완료: {copied}장  (source: {SRC})")
else:
    print(f"이미 준비됨({SRC} 기준): {len(os.listdir(DATA_DIR))}개 항목")

In [ ]:
# @title 이미지 종류 분리
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp")
CLASS_NAMES = ["이물질", "스크래치"]

CAUSE_RULES = {
    "_p": 0,          # 이물질 (particle)
    "이물": 0,
    "_s": 1,          # 스크래치 (scratch)
    "스크": 1,
}

def parse_filename(path):
    name = os.path.splitext(os.path.basename(path))[0].lower()
    m = re.match(r"^(\d+)[-_]([ox])(.*)$", name)
    if not m:
        return None
    num, ox, rest = m.groups()
    cause = None
    if ox == "x":
        for kw, cls in CAUSE_RULES.items():
            if kw in rest:
                cause = cls
                break
    return {
        "path": path,
        "num": int(num),
        "label": "good" if ox == "o" else "bad",
        "is_die": "die" in rest,
        "cause": cause,
    }

all_files = sorted(
    p for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)
    if p.lower().endswith(IMG_EXTS)
)

parsed   = [info for p in all_files if (info := parse_filename(p))]
unparsed = [p for p in all_files if parse_filename(p) is None]

die_files = [f for f in parsed if f["is_die"]]

cause_files   = [f for f in die_files if f["label"] == "bad" and f["cause"] is not None]
unknown_cause = [f for f in die_files if f["label"] == "bad" and f["cause"] is None]

cls_paths = [sorted(f["path"] for f in cause_files if f["cause"] == c)
             for c in range(len(CLASS_NAMES))]

print(f"전체 이미지          : {len(all_files)}장")
print(f"  ├ die 불량(원인 O) : {len(cause_files)}장  ->  "
      + " / ".join(f"{n} {len(p)}" for n, p in zip(CLASS_NAMES, cls_paths)))
print(f"  ├ die 불량(원인 ?) : {len(unknown_cause)}장   <- CAUSE_RULES 보완 필요")
print(f"  └ 규칙 불일치       : {len(unparsed)}장")

if unknown_cause:
    print("\n[확인 필요] 원인 키워드가 안 잡힌 불량 파일 예시:")
    for f in unknown_cause[:10]:
        print("   ", os.path.basename(f["path"]))

assert all(len(p) > 0 for p in cls_paths), "클래스별 이미지가 비어 있음 — CAUSE_RULES 확인"


In [ ]:
# @title 이미지 학습 / 검증 데이터 분리
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

IMG_SIZE = 256

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])

random.seed(42)
tr_by_cls, va_paths, va_labels = [], [], []
for c, paths in enumerate(cls_paths):
    paths = paths[:]
    random.shuffle(paths)
    split = int(len(paths) * 0.8)
    tr_by_cls.append(paths[:split])
    va_paths  += paths[split:]
    va_labels += [c] * (len(paths) - split)

class CyclicBalancedDataset(Dataset):
    def __init__(self, paths_by_class, transform):
        self.by_cls = [list(p) for p in paths_by_class]
        self.tf = transform
        self.chunk = min(len(p) for p in self.by_cls)
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.samples = []
        for c, paths in enumerate(self.by_cls):
            n = len(paths)
            start = (epoch * self.chunk) % n
            end = start + self.chunk
            sel = paths[start:end] if end <= n else paths[start:] + paths[:end - n]
            self.samples += [(p, c) for p in sel]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.tf(Image.open(path).convert("RGB")), label

class EvalDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.tf = paths, labels, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

train_ds = CyclicBalancedDataset(tr_by_cls, train_tf)
val_ds   = EvalDataset(va_paths, va_labels, eval_tf)
val_dl   = DataLoader(val_ds, batch_size=8)

print("학습(epoch당):", len(train_ds), "장 / 검증:", len(val_ds), "장")
print("검증 구성:", {CLASS_NAMES[c]: va_labels.count(c) for c in range(len(CLASS_NAMES))})


In [ ]:
# @title 자체 build_cnn 모델 구성
import torch.nn as nn

def build_cnn(channels=[16, 32, 64], num_classes=len(CLASS_NAMES), in_size=IMG_SIZE):
    layers, in_ch = [], 3
    for out_ch in channels:
        layers += [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        ]
        in_ch = out_ch
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, num_classes)]
    return nn.Sequential(*layers)

model = build_cnn([8, 16, 32]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"파라미터 수: {n_params:,}")

SyntaxError: incomplete input (3657487337.py, line 3)

In [ ]:
# @title 학습
import torch.nn as nn
from torch.utils.data import DataLoader

@torch.no_grad()
def evaluate(model, val_dl, device, n_classes=len(CLASS_NAMES)):
    """[수정] 클래스별 recall/precision -> macro 평균 (이물질/스크래치 동등 취급)"""
    model.eval()
    cm = torch.zeros(n_classes, n_classes, dtype=torch.long)
    for x, y in val_dl:
        pred = model(x.to(device)).argmax(1).cpu()
        for gt, pr in zip(y, pred):
            cm[gt, pr] += 1
    recalls    = [cm[c, c] / cm[c].sum() if cm[c].sum() else 0 for c in range(n_classes)]
    precisions = [cm[c, c] / cm[:, c].sum() if cm[:, c].sum() else 0 for c in range(n_classes)]
    acc = cm.diag().sum() / cm.sum()
    return acc.item(), float(sum(recalls) / n_classes), float(sum(precisions) / n_classes)

def train_model(model, train_ds, val_dl, device,
                epochs=30, lr=1e-3, batch_size=32,
                patience=7, save_path="best_classifier.pth"):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    best = {"recall": -1, "precision": -1, "acc": -1, "epoch": -1}
    no_improve, history = 0, []

    for ep in range(1, epochs + 1):
        train_ds.set_epoch(ep - 1)
        train_dl = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=2)

        model.train()
        tr_loss = 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(x)
        tr_loss /= len(train_ds)

        acc, recall, precision = evaluate(model, val_dl, device)
        history.append((tr_loss, acc, recall, precision))

        improved = (recall, precision) > (best["recall"], best["precision"])
        if improved:
            best.update(recall=recall, precision=precision, acc=acc, epoch=ep)
            torch.save(model.state_dict(), save_path)
            no_improve = 0
        else:
            no_improve += 1

        print(f"epoch {ep:2d} | loss {tr_loss:.4f} | acc {acc:.3f} "
              f"| macro recall {recall:.3f} | macro precision {precision:.3f}"
              + (" *" if improved else ""))

        if no_improve >= patience:
            print(f"-> {patience} epoch 개선 없음, 조기 종료")
            break

    print(f"\nbest (epoch {best['epoch']}): recall {best['recall']:.3f}, "
          f"precision {best['precision']:.3f}, acc {best['acc']:.3f} -> {save_path}")
    return history, best

history, best = train_model(model, train_ds, val_dl, DEVICE)



In [ ]:
# @title 그래프로 나타내기
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image

losses, accs, recalls, precisions = zip(*history)
epochs_x = range(1, len(losses) + 1)
best_ep = max(range(len(recalls)), key=lambda i: (recalls[i], precisions[i]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(epochs_x, losses, color="#D85A30", lw=2, marker="o", ms=3)
ax[0].set_title("train loss")
ax[0].set_xlabel("epoch")
ax[0].grid(alpha=0.3)

ax[1].plot(epochs_x, accs, color="#888880", lw=1.5, ls="--", label="accuracy")
ax[1].plot(epochs_x, recalls, color="#1D9E75", lw=2, marker="o", ms=3,
           label="macro recall")
ax[1].plot(epochs_x, precisions, color="#534AB7", lw=2, marker="s", ms=3,
           label="macro precision")
ax[1].axvline(best_ep + 1, color="#1D9E75", ls=":", alpha=0.6)
ax[1].annotate(f"best (ep {best_ep + 1})", xy=(best_ep + 1, recalls[best_ep]),
               xytext=(5, 8), textcoords="offset points", fontsize=9, color="#1D9E75")
ax[1].set_title("validation metrics")
ax[1].set_xlabel("epoch")
ax[1].set_ylim(0, 1.05)
ax[1].grid(alpha=0.3)
ax[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

model.load_state_dict(torch.load("best_classifier.pth"))
model.eval()

name_w = 40
print(f"{'파일':{name_w}s} 정답     예측     확률")
wrong = 0
with torch.no_grad():
    for path, gt_idx in zip(va_paths, va_labels):
        x = eval_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)
        probs = torch.softmax(model(x), 1)[0]
        pred_idx = probs.argmax().item()
        gt, pred = CLASS_NAMES[gt_idx], CLASS_NAMES[pred_idx]
        mark = "" if gt == pred else "  <- 오답"
        wrong += gt != pred
        print(f"{os.path.basename(path):{name_w}s} {gt:6s} {pred:6s} "
              f"{probs[pred_idx].item():.2f}{mark}")

print(f"\n오답 {wrong} / {len(va_paths)}")

In [ ]:
# @title Confusion Matrix
import torch
import numpy as np
import matplotlib.pyplot as plt

@torch.no_grad()
def confusion_matrix(model, val_dl, device, n_classes=len(CLASS_NAMES)):
    model.eval()
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for x, y in val_dl:
        pred = model(x.to(device)).argmax(1).cpu().numpy()
        for gt, pr in zip(y.numpy(), pred):
            cm[gt][pr] += 1
    return cm

def plot_confusion(cm, class_names=CLASS_NAMES):
    n = len(class_names)
    fig, ax = plt.subplots(figsize=(1.8 * n + 2, 1.8 * n + 1.5))
    ax.imshow(cm, cmap="Greens", vmin=0)
    for i in range(n):
        for j in range(n):
            correct = i == j
            ax.text(j, i, f"{cm[i][j]}", ha="center", va="center", fontsize=13,
                    fontweight="bold" if correct else "normal",
                    color="#04342C" if correct else "#791F1F")
    ax.set_xticks(range(n), [f"예측: {c}" for c in class_names])
    ax.set_yticks(range(n), [f"정답: {c}" for c in class_names])
    ax.set_title("confusion matrix (검증 셋)")
    plt.tight_layout()
    plt.show()

def print_metrics(cm):
    print(f"{'클래스':10s} {'recall':>8s} {'precision':>10s} {'개수':>6s}")
    for i, name in enumerate(CLASS_NAMES):
        r = cm[i][i] / cm[i].sum() if cm[i].sum() else 0
        p = cm[i][i] / cm[:, i].sum() if cm[:, i].sum() else 0
        print(f"{name:10s} {r:>8.3f} {p:>10.3f} {cm[i].sum():>6d}")
    off = cm.sum() - np.trace(cm)
    print(f"\n혼동(오분류) 총 {off}건")

cm = confusion_matrix(model, val_dl, DEVICE)
plot_confusion(cm)
print_metrics(cm)